# Large-Scale Exploratory Data Analysis of NYC Ferry Ridership

## Objective
Perform a large-scale exploratory data analysis on NYC Ferry ridership data to uncover
temporal, route-level, stop-level, and directional demand patterns using memory-efficient
data processing techniques.

## Dataset
- Source: NYC Open Data – NYC Ferry Ridership
- Size: ~2.74 million records
- Granularity: Hourly ridership by route, stop, and direction

## Approach
- Memory-aware data loading and preprocessing
- Aggregation-based exploratory analysis
- Insight-driven summaries suitable for downstream visualization (Tableau)


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Environment ready")


Environment ready


In [21]:
import plotly.io as pio
pio.renderers.default = "png"

In [3]:
DATA_PATH = "../data/raw/NYC_Ferry_Ridership.csv"
import os
import sys
sys.path.append(os.path.abspath(".."))
from  src.preprocessing import preprocess_data
df =preprocess_data(DATA_PATH)

In [4]:
from src.preprocessing import understand_data
understand_data(df)


DataFrame Shape: (2760352, 7)


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 2760352 entries, 0 to 2760653
Data columns (total 7 columns):
 #   Column     Dtype         
---  ------     -----         
 0   Date       datetime64[ns]
 1   Hour       Int8          
 2   Route      category      
 3   Direction  category      
 4   Stop       category      
 5   Boardings  Int32         
 6   TypeDay    category      
dtypes: Int32(1), Int8(1), category(4), datetime64[ns](1)
memory usage: 71.1 MB
None


DataFrame Description:
                                 Date         Hour    Route Direction  \
count                         2760352 2,760,352.00  2760352   2760352   
unique                            NaN         <NA>        9         2   
top                               NaN         <NA>       ER        SB   
freq                              NaN         <NA>   640481   1380915   
mean    2021-11-25 01:48:16.545367552        13.75      NaN       NaN   
min              

## Temporal Analysis

This section analyzes how NYC Ferry ridership varies over time, focusing on
hourly demand patterns and differences between weekdays and weekends.


In [5]:
hourly_ridership = (
    df.groupby("Hour",observed=True)["Boardings"]
    .sum()
    .reset_index()
    .sort_values("Hour")
)

hourly_ridership.head(20)

,Hour,Boardings
0,0,840
1,5,158841
2,6,671819
3,7,2274444
4,8,3176823
5,9,2354370
6,10,2160834
7,11,2722035
8,12,3108597
9,13,3347818


In [ ]:
fig = px.line(
    hourly_ridership,
    x="Hour",
    y="Boardings",
    markers=True,
    template="plotly_white", # Cleaner background
    color_discrete_sequence=["#F5CD05"] # NYC Ferry Blue-ish
)

fig.update_layout(
    title="<b>NYC Ferry Ridership by Hour of Day</b>",
    xaxis=dict(
        title="Hour of Day (24h)",
        tickmode="linear", 
        tick0=0, 
        dtick=3 # Shows a label every 3 hours
    ),
    yaxis=dict(
        title="Total Boardings"
    ),
    hovermode="x unified" # Shows all data for that hour when hovering
)

fig.show("png")

In [7]:
day_type_summary = (
    df.groupby("TypeDay", observed=True)["Boardings"]
      .agg(total_boardings="sum", average_boardings="mean")
      .reset_index()
)

day_type_summary

,TypeDay,total_boardings,average_boardings
0,Weekday,32197996,15.85
1,Weekend,16127786,22.13


In [ ]:
fig = px.bar(
    day_type_summary,
    x="TypeDay",
    y="total_boardings",
    color="TypeDay", # Different colors for each bar
    color_discrete_map={"Weekday": "#206EDC", "Weekend": "#F30A0A"} # Specific colors
)

fig.update_layout(
    showlegend=False, # Hides the legend since the x-axis already says the type
)
fig.show("png")

## Route-Level Analysis

This section examines how ridership is distributed across NYC Ferry routes
to identify high-demand routes and overall concentration patterns.


In [9]:
route_ridership = (
    df.groupby("Route", observed=True)["Boardings"]
      .sum()
      .reset_index()
      .sort_values("Boardings", ascending=False)
)

route_ridership.head(10)


,Route,Boardings
1,ER,18542143
0,AS,9781098
5,SB,5709224
4,RW,5700575
7,SV,5272249
6,SG,2231192
8,LE,563772
2,GI,483794
3,RR,41735


In [ ]:
top_routes = route_ridership.head(10)

# Sort the dataframe by Boardings before plotting
top_routes = top_routes.sort_values(by="Boardings", ascending=True) 

fig = px.bar(
    top_routes,
    y="Route",
    x="Boardings",
    orientation='h',
    text_auto='.2s', # Adds labels (e.g. 2.1M) inside/outside bars
    template="plotly_white",
    color_discrete_sequence=["#116EFA"] # NYC Ferry Blue-ish
)
fig.update_layout(
    title="<b>Top 10 NYC Ferry Routes by Ridership</b>",
    xaxis_title="Total Boardings",
    yaxis_title="", # Removing the label "Route" since it's obvious
    yaxis={'categoryorder':'total ascending'}, # Double-check sorting
    showlegend=False,
    coloraxis_showscale=False # Hides the color bar for a cleaner look
)
fig.show("png")


In [11]:
top_5_share = (
    route_ridership.head(5)["Boardings"].sum()
    / route_ridership["Boardings"].sum()
)
print(f"Top 5 routes account for {top_5_share:.2%} of total ridership.")

Top 5 routes account for 93.13% of total ridership.


## Stop-Level Analysis

This section analyzes ridership distribution across ferry stops to identify
high-traffic hubs and assess how demand is concentrated geographically.


In [12]:
stop_ridership = (
    df.groupby("Stop", observed=True)["Boardings"]
      .sum()
      .reset_index()
      .sort_values("Boardings", ascending=False)
)

stop_ridership.head(10)

,Stop,Boardings
27,Wall St/Pier 11,10803965
9,East 34th Street,6498148
18,North Williamsburg,3258963
8,Dumbo/Fulton Ferry,2598921
20,Rockaway,2548710
10,East 90th St,1988976
7,Dumbo/BBP Pier 1,1923146
16,Long Island City,1881509
15,Hunters Point South,1838632
0,Astoria,1780439


In [ ]:
top_stops = stop_ridership.head(15)

fig = px.bar(
    top_stops,
    y="Stop",
    x="Boardings",
    orientation='h',
    text_auto='.2s', # Adds labels (e.g. 2.1M) inside/outside bars
    template="plotly_white",
    color_discrete_sequence=["#12506F"] 
    
)

fig.update_layout(
    title="Top 15 NYC Ferry Stops by Ridership",
    xaxis_title="Total Boardings",
    yaxis_title="Stop",
     height=600, # Gives each stop a bit more breathing room
    yaxis={'categoryorder':'total ascending'} # Ensures the #1 stop is at the top
)
fig.show("png")

In [14]:
top_10_stop_share = (
    stop_ridership.head(10)["Boardings"].sum()
    / stop_ridership["Boardings"].sum()
)

print(f"Top 10 stops account for {top_10_stop_share:.2%} of total ridership.")

Top 10 stops account for 72.68% of total ridership.


## Directional Flow Analysis

This section examines ridership patterns by travel direction to assess
whether ferry demand is balanced or directionally skewed.


In [15]:
direction_ridership = (
    df.groupby("Direction", observed=True)["Boardings"]
      .sum()
      .reset_index()
)

direction_ridership

,Direction,Boardings
0,NB,23479363
1,SB,24846419


In [ ]:
fig = px.bar(
    direction_ridership,
    x="Direction",
    y="Boardings",
    template="plotly_white",
    color="Direction", # Different colors for each bar
    color_discrete_map={"NB": "#0E71FC", "SB": "#F30A0A"} # Specific colors
)

fig.update_layout(
    title="NYC Ferry Ridership by Direction",
    xaxis_title="Direction",
    yaxis_title="Total Boardings",
    showlegend=False # Hides the legend since the x-axis already says the direction
)
fig.show("png")



In [17]:
direction_ridership.assign(
    share=direction_ridership["Boardings"]
    / direction_ridership["Boardings"].sum()
)

,Direction,Boardings,share
0,NB,23479363,0.49
1,SB,24846419,0.51


## Key Insights

### 1. Strong Temporal Demand Peaks
Ferry ridership exhibits clear morning and evening peaks, reflecting commuter-driven usage patterns. Demand rises sharply after early morning hours, peaks in the late afternoon, and declines significantly during late-night periods.

### 2. Highly Concentrated Route Usage
Ridership is heavily concentrated across a small subset of ferry routes. The top five routes account for the majority of total boardings, highlighting uneven demand distribution across the network.

### 3. Stop-Level Demand Is Hub-Dominated
A limited number of ferry stops act as major demand hubs. Locations such as Wall St/Pier 11 and East 34th Street contribute disproportionately to overall ridership, indicating strong spatial centralization.


## EDA Summary & Next Steps

This exploratory analysis reveals that NYC Ferry ridership exhibits strong temporal structure, with clear long-term trends and recurring seasonal patterns across months and years. Ridership levels show persistence over time, indicating that past demand is informative of future demand.

Route-level analysis suggests that a subset of routes contributes disproportionately to total ridership, while others display higher volatility, motivating aggregation strategies for stable forecasting. Additionally, external shocks such as the COVID-19 period introduce temporary disruptions, followed by a gradual stabilization phase.

Overall, the presence of trend, seasonality, and temporal dependence supports the use of time-series forecasting approaches. Based on these findings, the next phase of the project focuses on feature engineering (lagged values, calendar effects, and rolling statistics) and predictive modeling, which are handled in a separate notebook.


In [18]:
os.makedirs("../data/processed", exist_ok=True)

# Export temporal summaries
hourly_ridership.to_csv(
    "../data/processed/hourly_ridership.csv",
    index=False
)

day_type_summary.to_csv(
    "../data/processed/day_type_summary.csv",
    index=False
)

# Export route and stop summaries
route_ridership.to_csv(
    "../data/processed/route_ridership.csv",
    index=False
)

stop_ridership.to_csv(
    "../data/processed/stop_ridership.csv",
    index=False
)

# Export directional summary (cleaned)
direction_ridership.to_csv(
    "../data/processed/direction_ridership.csv",
    index=False
)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


In [19]:
OUTPUT_PATH = "../data/processed/cleaned_ferry_ridership.csv"
from src.preprocessing import saving_cleaned_data
saving_cleaned_data(df, OUTPUT_PATH)

Cleaned dataset saved to ../data/processed/cleaned_ferry_ridership.csv
